# Pulldown Y SNPs with OY SNPs

In [1]:
import numpy as np
import os  # For Saving to Folder
import pandas as pd
import matplotlib.pyplot as plt

import socket
import os as os
import sys as sys
import multiprocessing as mp
from pysam import AlignmentFile

### For Arial Font
from matplotlib import rcParams
rcParams['font.family'] = 'sans-serif'   # Set the defaul
### Make sure to have the font installed (it is on cluster for Harald)
rcParams['font.sans-serif'] = ['Arial']

socket_name = socket.gethostname()
print(socket_name)

if socket_name.startswith("compute-"):
    print("HSM Computational partition detected.")
    path = "/n/groups/reich/hringbauer/git/y_chrom/"  # The Path on Midway Cluster
    
elif socket_name.startswith("bionc") or socket_name.startswith("hpc"):
    print("Leipzig Cluster detected!")
    path = "/mnt/archgen/users/hringbauer/git/y_chrom/"
    
else:
    raise RuntimeWarning("Not compatible machine. Check!!")

os.chdir(path)  # Set the right Path (in line with Atom default)

# Show the current working directory. Should be HAPSBURG/Notebooks/ParallelRuns
print(os.getcwd())
print(f"CPU Count: {mp.cpu_count()}")
print(sys.version)

### Custom Imports
from python.pulldown import load_snp_file_ISOGG, call_y_bam, mismatch_path, div_anc_der, get_mismatch_snps, create_parent_dct

hpc030
Leipzig Cluster detected!
/mnt/archgen/users/hringbauer/git/y_chrom
CPU Count: 128
3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]


### [1x Requirement Done]
Prepear OY bed file (available)

### 0) Load data

In [2]:
### Load OY SNPs
df1 = pd.read_csv("/mnt/archgen/users/hringbauer/git/y_chrom/data/all_snps_filtered_levels.csv", low_memory=False)
print(f"Loaded {len(df1)} OY SNPs with levels loaded")

### Load OY node dictionary
chpar = create_parent_dct()

### Load "faulty" SNPs to exclude from summary stats
df_cts = pd.read_csv("/mnt/archgen/users/eric_garcia/OYdb/mm12_v3.tsv", sep="\t")
df_ex = df_cts[df_cts["count"]>5] # SNPs that are ancestral in at least 3/12 test sample derived chains

Loaded 2868884 OY SNPs with levels loaded


### 0b) Functions for calling the haplogroup

In [150]:
def call_OY(bam_path="", df=[], df_ex=[], path_bed='',
            path_temp='', snip5=0, snip3=0, score_factor=3):
    """Call Y Haplogroup using OY SNPs.
    score_factor: Penalty of ancestral SNPs"""

    ### Default for MPI EVA [todo: Include into package]
    if len(df)==0:
        df = pd.read_csv("/mnt/archgen/users/hringbauer/git/y_chrom/data/all_snps_filtered_levels.csv", low_memory=False)
        print(f"Loaded {len(df)} OY SNPs with levels loaded")
        
    if len(path_temp)==0:
        path_temp = "/mnt/archgen/users/hringbauer/git/y_chrom/temp/temp.tsv"

    if len(path_bed)==0:
        path_bed= "/mnt/archgen/users/hringbauer/git/y_chrom/data/OY_snps.bed"

    ####### Y Haplogroup Calling
    ### Run the pulldown
    df_ch, df_der = call_y_bam(path_bam=bam_path, df=df, path_bed=path_bed, 
                               snip5=snip5, snip3=snip3, path_temp=path_temp) 

    ### Post-process output for info for manual call
    dft = div_anc_der(df_ch, df_exclude=df_ex) # Summary per haplogroup
    dfd = dft[dft["Derived"]>dft["Ancestral"]].copy() # Only derived ones

    ### Add Scores
    dfd["Score"] = (dfd["#ANC in par."]*-score_factor)+dfd["#DER in par."] + (dfd["Ancestral"]*-score_factor)+dfd["Derived"]
    dfd = dfd.sort_values(by="Score", ascending=True) # Sort by Score
    y_haplo_call = dfd["Branch"].values[-1]
    
    return y_haplo_call, dfd, df_ch, df_der

def call_OYs(bam_paths=[], iids=[], df=[], df_ex=[], path_bed='',
            path_temp='', snip5=0, snip3=0, score_factor=3):
    """Call Y haplogroups for multiple bam paths.
    Return a dataframe with calls (point calls only). 
    For detailed output, run call_OY on single samples"""
    
    assert(np.all([os.path.exists(p) for p in bam_paths])) # Quick Sanity Check of Input BAMs

    calls = []
    n = len(bam_path)
    for i, bam_path in enumerate(bam_paths):
        print(f"Running {i+1}/{n}: {bam_path}")
        _, dfd, _, _ = call_OY(bam_path=bam_path, df=df, df_ex=df_ex, path_bed=path_bed,
                path_temp=path_temp, snip5=snip5, snip3=snip3, score_factor=score_factor)
        calls += [dfd[-1:]] # The most derived output
        
    # Format Output
    df_res = pd.concat(calls)
    if len(iids)>0:
        assert(len(iids)==len(df_res))
        df_res["iid"]=iids

    return df_res

In [96]:
%%time
bam_paths = ["/mnt/archgen/users/hringbauer/data/temp/I24680.bam", 
             "/mnt/archgen/Autorun_eager/eager_outputs/SG/BMG/BMG001/merged_bams/additional/BMG001_ss_libmerged_add.bam"]
df_res = call_OYs(bam_paths, df=df1, df_ex=df_ex)

Average Coverage: 0.1773x
#Sites covered: 281824/2868884
#Derived Loci: 
1927 / 281824 covered>0
Average Coverage: 0.3665x
#Sites covered: 824045/2868884
#Derived Loci: 
3575 / 824045 covered>0
CPU times: user 3.29 s, sys: 29.6 ms, total: 3.32 s
Wall time: 19.9 s


In [97]:
df_res

,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.,Score
19963,R-BY180206,37,1,0,1,0,1,189,187
48477,R-FTA63331,46,1,0,1,0,2,246,241


In [94]:
dfd[-1:]

,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.,Score
2953,R-L151,31,1,0,1,0,0,21,22


# 1) Example BAM file
Use downloaded BAM file from Punic project

In [54]:
y_haplo_call, dfd, df_ch, df_der = call_OY(bam_path="/mnt/archgen/users/hringbauer/data/temp/I24680.bam", df=df1, df_ex=df_ex)

Average Coverage: 0.1773x
#Sites covered: 281824/2868884
#Derived Loci: 
1927 / 281824 covered>0


In [64]:
dfd[-10:]

,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.,Score
24947,R-FT158050,51,1,0,1,0,7,187,167
31047,R-FTE76346,44,1,0,1,0,6,187,170
23704,R-FGC15048,39,1,0,1,0,6,187,170
21332,R-BY3983,43,1,0,1,0,6,187,170
18859,R-BY108187,46,1,0,1,0,6,187,170
30304,R-FTD44445,39,1,0,1,0,2,187,182
32087,R-M269,27,38,0,38,0,0,148,186
34048,R-Y88944,36,2,0,2,0,1,187,186
32028,R-L23,28,1,0,1,0,0,186,187
19963,R-BY180206,37,1,0,1,0,1,189,187


In [49]:
y_haplo_call

'C-F11120'

In [59]:
dfmm = get_mismatch_snps('R-BY180206', chpar=chpar, df_ch=df_ch)
dfmm[~dfmm["Subgroup Name"].isin(df_ex["Subgroup Name"])]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
274416,FT20722,Y,13859387,G,A,R-Z2110,NaN,33,0,0,1,0,1,0


In [63]:
df_ch[df_ch["Y-haplogroup"]=="R-Z2110"]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
274416,FT20722,Y,13859387,G,A,R-Z2110,NaN,33,0,0,1,0,1,0


# 1b) Run Henry II

In [67]:
%%time
#path_bam = "/mnt/archgen/Autorun/Results/Human_Shotgun/260319_LH00454_0192_A23CV73LT4/WSQ003.A0101.SG1.3/out.bam"
path_bam =  "/mnt/archgen/Autorun_eager/eager_outputs/SG/BMG/BMG001/merged_bams/additional/BMG001_ss_libmerged_add.bam"
y_haplo_call, dfd, df_ch, df_der = call_OY(bam_path=path_bam, df=df1, df_ex=df_ex)

Average Coverage: 0.3665x
#Sites covered: 824045/2868884
#Derived Loci: 
3575 / 824045 covered>0
CPU times: user 2.1 s, sys: 111 ms, total: 2.21 s
Wall time: 12.2 s


In [68]:
dfd[-20:]

,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.,Score
55346,R-S12083,40,1,0,1,0,7,237,217
40110,R-FGC17165,41,1,0,1,0,5,237,223
33909,R-BY1701,41,1,0,1,0,4,237,226
56262,R-Y157757,44,1,0,1,0,4,237,226
44811,R-FT309315,42,1,0,1,0,4,237,226
44078,R-FT251688,45,1,0,1,0,4,237,226
41080,R-FGC68456,45,1,0,1,0,4,237,226
54960,R-M269,27,36,1,35,0,1,198,227
36344,R-BY3953,45,1,0,1,0,3,237,229
54877,R-L196,46,1,0,1,0,3,237,229


In [69]:
y_haplo_call

'R-FTA63331'

In [70]:
df_ch[df_ch["Y-haplogroup"]=="R-FTA63879"]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
846849,FTA65919,Y,22139468,T,C,R-FTA63879,NaN,45,0,1,0,0,0,1
849672,FTA63879,Y,14861556,T,A,R-FTA63879,NaN,45,1,0,0,0,0,1


In [71]:
dfmm = get_mismatch_snps('R-FTA63331', chpar=chpar, df_ch=df_ch)
dfmm[~dfmm["Subgroup Name"].isin(df_ex["Subgroup Name"])]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
747336,"FTA9712,Y137",Y,7451254,A,G,R-M269,NaN,27,1,0,0,0,1,0
704506,"FGC26653,YP1981",Y,16393584,T,C,A-L1090,NaN,2,0,0,0,1,1,0


### 1c) Run Brienzi

In [73]:
%%time
path_bam="/mnt/archgen/users/hringbauer/data/brienziYcapture/A55903.bam" #A55903 and A55904

y_haplo_call, dfd, df_ch, df_der = call_OY(bam_path=path_bam, df=df1, df_ex=df_ex)

Average Coverage: 10.3055x
#Sites covered: 1636489/2868884
#Derived Loci: 
3951 / 1636489 covered>0
CPU times: user 3.19 s, sys: 291 ms, total: 3.48 s
Wall time: 32.2 s


In [74]:
dfd[-20:]

,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.,Score
38228,P-M45,17,5,0,5,0,0,295,300
38230,P-P284,18,4,0,4,0,0,300,304
38229,P-P226,19,32,0,32,0,0,304,336
71035,R-UTY2,20,31,0,31,0,0,336,367
69929,R-L389,25,2,0,2,0,0,367,369
70163,R-P25,22,8,0,7,1,0,367,374
70165,R-P297,26,15,1,14,0,0,369,380
41532,R-BY1188,53,1,0,1,0,15,453,409
45908,R-BY3293,35,1,0,1,0,13,449,411
44514,R-BY19415,46,1,0,1,0,12,453,418


In [75]:
%%time
path_bam="/mnt/archgen/users/hringbauer/data/brienziYcapture/A55904.bam" #A55903 and A55904

y_haplo_call, dfd, df_ch, df_der = call_OY(bam_path=path_bam, df=df1, df_ex=df_ex)

Average Coverage: 8.4978x
#Sites covered: 1558116/2868884
#Derived Loci: 
3785 / 1558116 covered>0
CPU times: user 3.11 s, sys: 100 ms, total: 3.21 s
Wall time: 27.4 s


In [76]:
dfd[-10:]

,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.,Score
72875,R-Y85206,47,1,0,1,0,8,437,414
54359,R-FT22796,42,1,0,1,0,5,436,422
68430,R-M269,27,63,1,62,0,1,368,424
68341,R-L23,28,1,0,1,0,2,430,425
68361,R-L51,29,2,0,2,0,2,431,427
68330,R-L151,31,3,0,3,0,2,433,430
49618,R-DF27,35,1,0,1,0,2,436,431
45092,R-BY3508,38,1,0,1,0,2,436,431
56377,R-FT355076,39,2,0,2,0,2,437,433
64444,R-FTD25777,41,11,0,11,0,2,439,444


In [ ]:
# https://discover.familytreedna.com/y-dna/R-FTD25777/story

In [77]:
%%time
path_bam="/mnt/archgen/users/hringbauer/data/brienziYcapture/gander.john.bam" #A55903 and A55904

y_haplo_call, dfd, df_ch, df_der = call_OY(bam_path=path_bam, df=df1, df_ex=df_ex)

Average Coverage: 1.0741x
#Sites covered: 15216/2868884
#Derived Loci: 
105 / 15216 covered>0
CPU times: user 527 ms, sys: 37 ms, total: 564 ms
Wall time: 3.96 s


In [79]:
dfd[-10:]

,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.,Score
1416,K-M9,12,2,0,2,0,0,10,12
1415,K-M526,13,1,0,1,0,0,12,13
1691,P-M45,17,1,0,1,0,0,13,14
3023,R-UTY2,20,1,0,1,0,0,14,15
2964,R-L389,25,1,0,1,0,0,15,16
1946,R-BY203122,41,1,0,1,0,2,22,17
2992,R-P297,26,2,0,2,0,0,16,18
2981,R-M269,27,2,0,2,0,0,18,20
2967,R-L51,29,1,0,1,0,0,20,21
2953,R-L151,31,1,0,1,0,0,21,22


In [80]:
dfmm = get_mismatch_snps("R-FTD25777", chpar=chpar, df_ch=df_ch)
dfmm[~dfmm["Subgroup Name"].isin(df_ex["Subgroup Name"])]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#


## 1c) High-Coverage PTN sample

In [74]:
%%time
path_bam = "/mnt/archgen/Autorun_eager/eager_outputs/SG/PTN/PTN209/trimmed_bam/PTN209_ss_libmerged_udghalf.trimmed.bam"

df_ch, df_der = call_y_bam(df=df1, path_bam=path_bam,
                           path_bed='/mnt/archgen/users/hringbauer/git/y_chrom/data/OY_snps.bed') 

Average Coverage: 7.2976x
#Sites covered: 2370738/2868884
#Derived Loci: 
6390 / 2370738 covered>0
CPU times: user 2.54 s, sys: 1.74 s, total: 4.27 s
Wall time: 51.7 s


In [ ]:
df_der.sort_values(by="Level")[-50:]

In [88]:
dft = div_anc_der(df_ch)
dfd =dft[dft["Derived"]>dft["Ancestral"]]
dfd.sort_values(by="#DER in par.").tail(20)

,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.
84668,R-M269,27,97,0,97,0,29,589
1448,C-V9,7,38,3,35,0,286,656
84564,R-L23,28,3,0,3,0,29,686
84587,R-L51,29,5,0,5,0,29,689
55893,R-BY3293,35,1,0,1,0,43,689
84552,R-L151,31,3,0,3,0,29,694
84890,R-PF6538,31,1,0,1,0,29,694
50798,R-BY1188,53,1,0,1,0,52,697
84558,R-L2,36,1,0,1,0,29,697
56679,R-BY3953,45,1,0,1,0,39,697


In [83]:
df_ch[df_ch["Y-haplogroup"]=="R-FTA63879"]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
2437353,FTA65919,Y,22139468,T,C,R-FTA63879,NaN,45,0,0,0,7,7,0
2441200,"FTA65732,PR6621",Y,21593639,T,C,R-FTA63879,NaN,45,0,0,0,13,13,0
2445506,FTA63879,Y,14861556,T,A,R-FTA63879,NaN,45,0,0,0,11,11,0


In [84]:
df_mms = get_mismatch_snps("R-FTG53091", chpar=chpar, df_ch=df_ch)

In [87]:
df_mms[:]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
2127395,"MF784814,YSC0000166",Y,14116584,A,T,R-P297,NaN,26,17,0,0,0,17,0
2090274,"FGC58,MF803532",Y,7100362,T,C,R-M343,NaN,22,0,0,0,8,8,0
2090443,"FGC66,MF803527",Y,7081561,T,C,R-M343,NaN,22,0,0,0,9,9,0
2073713,"FGC280,MF786133",Y,19298321,A,G,R-UTY2,NaN,20,9,0,0,0,9,0
2075838,"MF789201,YSC0000067",Y,7133986,C,G,R-UTY2,NaN,20,0,8,0,0,8,0
2037704,FGC222,Y,28699018,G,A,IJK-L15,IJK,11,0,0,9,0,9,0
2039873,"MF796482,TY198493,V1295",Y,7629583,G,A,IJK-L15,IJK,11,0,0,9,0,9,0
2030716,FGC2646,Y,14565310,A,C,F-M89,F,8,10,0,0,0,10,0
2031171,CTS5750,Y,16467111,T,C,F-M89,F,8,0,0,0,6,6,0
2032012,"MF808158,PF1911",Y,23729951,T,C,F-M89,F,8,0,0,0,7,7,0


# Other St. Pölten males 1x WGS

In [7]:
%%time
path_bam = "/mnt/archgen/Autorun_eager/eager_outputs/SG/PTN/PTN267/trimmed_bam/PTN267_ss.A0101_udghalf.trimmed.bam"

df_ch, df_der = call_y_bam(df=df1, path_bam=path_bam,
                           path_bed='/mnt/archgen/users/hringbauer/git/y_chrom/data/OY_snps.bed') 

Average Coverage: 0.4835x
#Sites covered: 1004319/2868884
#Derived Loci: 
4110 / 1004319 covered>0
CPU times: user 1.42 s, sys: 549 ms, total: 1.96 s
Wall time: 26.1 s


In [8]:
dft = div_anc_der(df_ch)
dfd =dft[dft["Derived"]>dft["Ancestral"]]
dfd.sort_values(by="#DER in par.").tail(20)

,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.
23932,J-FTD43091,36,1,0,1,0,67,197
26001,J-Y3082,33,1,0,1,0,55,197
16557,I-M170,14,46,1,45,0,14,197
25985,J-Y304060,34,1,0,1,0,105,197
22844,J-FTA47014,29,1,0,1,0,72,197
24460,J-FTF47360,30,1,0,1,0,31,197
16877,I-S31,15,14,0,14,0,15,242
10286,I-BY167444,24,1,0,1,0,33,242
11657,I-CTS2257,16,10,0,10,0,15,256
16878,I-S33,18,21,1,20,0,15,266


In [11]:
df_mms = get_mismatch_snps("I-FT58623", chpar=chpar, df_ch=df_ch)
df_mms

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
876403,"L37,PF6900,S153",Y,17516123,T,C,I-S33,NaN,18,0,0,0,1,1,0
869930,Y1934,Y,8504226,G,A,I-M170,I,14,0,0,1,0,1,0
867820,PF3518,Y,6607318,C,T,IJ-P124,NaN,12,0,2,0,0,2,0
868010,PF3561,Y,21389837,G,A,IJ-P124,NaN,12,0,0,1,0,1,0
866516,FGC222,Y,28699018,G,A,IJK-L15,IJK,11,0,0,1,0,1,0
867413,"MF796482,TY198493,V1295",Y,7629583,G,A,IJK-L15,IJK,11,0,0,3,0,3,0
863674,FGC2646,Y,14565310,A,C,F-M89,F,8,2,0,0,0,2,0
864223,"MF808158,PF1911",Y,23729951,T,C,F-M89,F,8,0,0,0,1,1,0
864239,"MF806436,PF1720,TY199699",Y,17142068,T,A,F-M89,F,8,0,0,0,2,2,0
861566,"V6478,Z9327",Y,17028360,T,A,A-V168,NaN,3,0,0,0,1,1,0


### Son of above sample

In [15]:
%%time
path_bam = "/mnt/archgen/Autorun_eager/eager_outputs/SG/PTN/PTN139/trimmed_bam/PTN139_ss_libmerged_udghalf.trimmed.bam"

df_ch, df_der = call_y_bam(df=df1, path_bam=path_bam,
                           path_bed='/mnt/archgen/users/hringbauer/git/y_chrom/data/OY_snps.bed') 

Average Coverage: 0.3297x
#Sites covered: 752016/2868884
#Derived Loci: 
3882 / 752016 covered>0
CPU times: user 1.24 s, sys: 493 ms, total: 1.73 s
Wall time: 19.5 s


In [16]:
dft = div_anc_der(df_ch)
dfd =dft[dft["Derived"]>dft["Ancestral"]]
dfd.sort_values(by="#DER in par.").tail(20)

,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.
48105,R-FTC3117,55,1,0,1,0,94,147
11214,I-FT235980,32,1,0,1,0,18,185
14769,I-S31,15,8,0,8,0,10,185
10477,I-FGC88432,23,1,0,1,0,18,185
15658,I-Y36690,33,1,0,1,0,19,185
10389,I-FGC52744,46,1,0,1,0,21,185
10212,I-CTS2257,16,4,0,4,0,10,193
14770,I-S33,18,14,1,13,0,10,197
9415,I-BY3095,29,1,0,1,0,40,210
14864,I-Y10720,20,2,0,2,0,11,210


In [17]:
df_mms = get_mismatch_snps("I-FT58623", chpar=chpar, df_ch=df_ch)
df_mms

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
655590,BY31307,Y,13690483,G,T,I-S33,NaN,18,0,0,1,0,1,0
648726,"MF796482,TY198493,V1295",Y,7629583,G,A,IJK-L15,IJK,11,0,0,2,0,2,0
646057,CTS5750,Y,16467111,T,C,F-M89,F,8,0,0,0,1,1,0
646332,"MF806436,PF1720,TY199699",Y,17142068,T,A,F-M89,F,8,0,0,0,1,1,0
646758,"CTS9317,MF806850,PF1767",Y,18818812,T,C,F-M89,F,8,0,0,0,5,5,0
644359,Z9315,Y,16933354,A,C,A-V168,NaN,3,1,0,0,0,1,0
644181,FGC27824,Y,23139472,T,C,A-L1090,NaN,2,0,0,0,1,1,0
644233,V1615,Y,8098483,T,A,A-L1090,NaN,2,0,0,0,1,1,0
644235,FGC26134,Y,13832165,G,T,A-L1090,NaN,2,0,0,2,0,2,0
644239,V3167,Y,15833573,T,C,A-L1090,NaN,2,0,0,0,1,1,0


### Hettstett male (Twist)
Test how high-coverage Twist sample performs when Y calling with OY SNPs

In [32]:
%%time
#path_bam = "/mnt/archgen/Autorun_eager/eager_outputs/RM/HET/HET022/trimmed_bam/HET022_ss.A0201_udgnone.trimmed.bam"
#path_bam = "/mnt/archgen/Autorun_eager/eager_outputs/RM/HET/HET024/trimmed_bam/HET024_ss.A0201_udgnone.trimmed.bam"
path_bam = "/mnt/archgen/Autorun_eager/eager_outputs/RM/HET/HET036/trimmed_bam/HET036_ss.A0201_udgnone.trimmed.bam"

df_ch, df_der = call_y_bam(df=df1, path_bam=path_bam,
                           path_bed='/mnt/archgen/users/hringbauer/git/y_chrom/data/OY_snps.bed') 

### Post-process output for info for manual call
dft = div_anc_der(df_ch, df_exclude=df_ex)
dfd = dft[dft["Derived"]>dft["Ancestral"]]
display(dfd.sort_values(by="#DER in par.").tail(30))

Average Coverage: 2.2975x
#Sites covered: 1156923/2868884
#Derived Loci: 
8127 / 1156923 covered>0


,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.
10833,I-BY141663,38,1,0,1,0,24,312
10177,I-A21912,33,1,0,1,0,20,312
13516,I-FT207349,36,1,0,1,0,23,312
12866,I-FGC89366,35,1,0,1,0,19,312
11776,I-BY44946,31,1,0,1,0,17,312
12524,I-CTS2257,16,9,0,9,0,0,338
11998,I-BY61070,34,1,0,1,0,61,347
19133,I-Y3945,34,1,0,1,0,36,347
12716,I-FGC30452,35,1,0,1,0,61,347
19023,I-Y359472,25,1,0,1,0,10,347


CPU times: user 2.66 s, sys: 79.6 ms, total: 2.74 s
Wall time: 24.1 s


In [33]:
dfmm = get_mismatch_snps("I-Y176112", chpar=chpar, df_ch=df_ch)
dfmm[~dfmm["Subgroup Name"].isin(df_ex["Subgroup Name"])]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#


# 2) Run lists of IIDs

### Hettstett IIDs

In [135]:
dfm = pd.read_csv("/mnt/archgen/users/hringbauer/git/auto_popgen/output/dumpster/stats_HET_RM.tsv", sep="\t")
print(f"Loaded {len(dfm)} iids")

dfm = dfm[dfm["snps_covered"]>100000]
dfmm = dfm[dfm["SexDetErrmine_mqc-generalstats-sexdeterrmine-RateY"]>0.15] # Get males
print(f"{len(dfmm)} males above coverage filter.")

### Intersect with Pandora Meta
#df_p = pd.read_csv("/mnt/archgen/users/hringbauer/git/auto_popgen/data/meta/pandora_meta.v0.18.tsv", sep="\t")
df_p = pd.read_csv("/mnt/archgen/users/hringbauer/git/auto_popgen/output/v0.3/master_df.tsv", sep="\t")
idx =df_p["iid"].isin(dfmm["Sample"].str.split("_").str[0])
df_run = df_p[idx].copy()

### Extract relevant Data Type
df_run1 = df_run[df_run["type"]=="RM"]
print(f"{len(df_run1)} in Autorun_Eager meta.")

Loaded 32 iids
13 males above coverage filter.
13 in Autorun_Eager meta.


In [145]:
bam_paths = df_run1["bam_path"][:2]
iids = df_run1["iid"].values[:2]

df_res = call_OYs(bam_paths=bam_paths, iids=iids, df=df1, df_ex=df_ex)

Average Coverage: 3.1375x
#Sites covered: 1219425/2868884
#Derived Loci: 
5649 / 1219425 covered>0
Average Coverage: 3.5503x
#Sites covered: 1328770/2868884
#Derived Loci: 
6654 / 1328770 covered>0


In [147]:
print("Finished!")

Finished!


In [148]:
df_res

,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.,Score,iid
50037,R-FT118626,51,1,0,1,0,0,441,442,HET038
14005,I-FT162921,35,1,0,1,0,0,429,430,HET022


In [144]:
iids

32533    HET038
32536    HET022
Name: iid, dtype: object